In [0]:
import pyspark.sql.functions as f
from pyspark.sql.functions import lit, col
import json
import time, ast

##extract using etteration

In [0]:

def normalize_pk_dict(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x

    s = str(x).strip()

    # JSON list or Python-list string
    if (s.startswith("[") and s.endswith("]")):
        try:
            v = json.loads(s)
            return v if isinstance(v, list) else [v]
        except Exception:
            v = ast.literal_eval(s)
            return v if isinstance(v, list) else [v]

    # single table name
    return [s]


raw = dbutils.widgets.get("pk_dict")  
pk_dict = normalize_pk_dict(raw)

print("pk_dict:", pk_dict, type(pk_dict))


In [0]:
g_ucBronze = "dev_hub_bronze"
g_ucSilver = "dev_hub_silver"
v_p_srcSchema = "lh_ax_idr"
META_TBL = "dev_bronze.poc._meta"
TS_COL   = "data_received_utc_dttm"

In [0]:
p_maxWorkers = 4


In [0]:
from pyspark.sql import functions as F

def sync_idr_tables(pk_dict, g_ucBronze, g_ucSilver, v_p_srcSchema, meta_fqn):
    """
    pk_dict: list[str] of table names, e.g. ["tcrstat"]
    meta_fqn: fully qualified meta table, e.g. "dev_bronze.poc._meta"
              must contain columns: TABLE_NM, PK_COL_NM
    """

    # 1) meta once
    df_meta = spark.table(meta_fqn)

    # 2) loop tables in pk_dict
    for table in pk_dict:
        # --- (A) check table exists in information_schema ---
        df_inf = spark.sql(f"""
            SELECT table_name
            FROM {g_ucSilver}.information_schema.tables
            WHERE table_catalog = '{g_ucSilver}'
              AND table_schema  = '{v_p_srcSchema}'
              AND lower(table_name) = lower('{table}')
        """)

        table_names = [r["table_name"] for r in df_inf.collect()]
        if not table_names:
            print(f"[SKIP] {table} not found in {g_ucSilver}.{v_p_srcSchema}")
            continue

        # usually 1 match
        for table_name in table_names:
            table_u = table_name.upper()

            # --- (B) get PK columns from meta ---
            pk_cols = (
                df_meta
                .filter(F.col("TABLE_NM") == table_u)
                .select("PK_COL_NM")
                .distinct()
                .collect()
            )
            pk_cols = [r["PK_COL_NM"] for r in pk_cols if r["PK_COL_NM"]]

            if not pk_cols:
                raise ValueError(f"No PK_COL_NM found in meta for TABLE_NM={table_u}")

            pk_cols_str = ", ".join([f"`{c}`" for c in pk_cols])

            # --- (C) load source and dedupe ---
            src_fqn = f"{g_ucBronze}.{v_p_srcSchema}.{table_name}"
            sdf = spark.table(src_fqn)
            sdf.createOrReplaceTempView("sdf")

            deduped = spark.sql(f"""
                SELECT * EXCEPT (rn)
                FROM (
                    SELECT *,
                           ROW_NUMBER() OVER (
                               PARTITION BY {pk_cols_str}
                               ORDER BY data_received_utc_dttm DESC
                           ) AS rn
                    FROM sdf
                )
                WHERE rn = 1
            """)

            deduped.createOrReplaceTempView("final")

            # --- (D) build MERGE ---
            columns = deduped.columns
            columns_str = ", ".join([f"`{c}`" for c in columns])
            values_str  = ", ".join([f"s.`{c}`" for c in columns])

            on_clause = " AND ".join([f"t.`{c}` = s.`{c}`" for c in pk_cols])

            non_pk_cols = [c for c in columns if c not in set(pk_cols)]
            update_set = ", ".join([f"t.`{c}` = s.`{c}`" for c in non_pk_cols])

            # If everything is PK (rare), do a harmless no-op update
            if not update_set:
                update_set = ", ".join([f"t.`{c}` = s.`{c}`" for c in pk_cols])

            tgt_fqn = f"{g_ucSilver}.{v_p_srcSchema}.{table_name}"

            merge_q = f"""
                MERGE INTO {tgt_fqn} t
                USING final s
                ON {on_clause}
                WHEN MATCHED THEN UPDATE SET {update_set}
                WHEN NOT MATCHED THEN INSERT ({columns_str}) VALUES ({values_str})
            """

            spark.sql(merge_q)
            print(f"[OK] {table_name} merged (PKs: {pk_cols})")

In [0]:

start_time = time.time()
sync_idr_tables(
    pk_dict=pk_dict,
    g_ucBronze=g_ucBronze,
    g_ucSilver=g_ucSilver,
    v_p_srcSchema=v_p_srcSchema,
    meta_fqn="dev_bronze.poc._meta"
    )
end_time = time.time()
print(f"Execution time: {end_time - start_time:.2f} seconds")